# 04 — Cultivability Candidate Ranking

Apply the trained classifier to all 235,671 HQ uncultured-or-environmental genomes, then filter to actionable cultivation candidates.

## Two candidate pools

**Lenient filter** (the broad recommendation list):
- Uncultured genome (`ncbi_genome_category != 'none'`)
- P(isolate) ≥ 0.5
- Family already has ≥1 cultured isolate (so the model has seen this family's isolate signature)
- Phylum is NOT a known auxotroph-rich CPR/DPANN-style lineage (avoid known-hard targets)

**Strict filter** (the genuinely-novel-target list):
- All of the above PLUS
- **The genus has zero isolates** — this is a genuinely uncultured lineage

Tiers: **S** (P ≥ 0.9), **A** (0.7 ≤ P < 0.9), **B** (0.5 ≤ P < 0.7).

In [1]:
!python ../src/rank_candidates.py 2>&1 | grep -E '(Candidate pool|Strict candidate|S-tier env)' | head

Lenient candidate pool: 2,491 MAGs (S=395, A=903, B=1193) across 13 phyla
Strict candidate pool:   256 MAGs (S= 19, A= 65, B= 172) across 21 phyla, 68 families, 118 genera
H3 enrichment: S-tier env-of-interest Fisher OR=4.44, p=3.6e-27


## Why two pools? — the labeling quirk

`ncbi_genome_category='derived from metagenome'` describes **assembly provenance**, not species cultivability. Many top scorers in the lenient pool are MAG-assembled genomes of *species that are well known to culture* — e.g., `Klebsiella pneumoniae`, `Pseudomonas E protegens`, `Phocaeicola dorei`, `Vibrio diabolicus`. These are correctly identified by the model as isolate-like, but they are not novel cultivation targets — somebody already cultured the species, just from a different sample.

The **strict** filter (`genus has zero isolates`) removes this confounder: 256 MAGs from 118 genuinely uncultured genera that sit phylogenetically inside families with cultured relatives. These are the actionable list.

## Lenient pool — top 15 by P(isolate)

```
      genome_id  p_isolate tier             phylum                family                         species
GCA_023639195.1     0.9999    S  p__Pseudomonadota   f__Pseudomonadaceae      s__Pseudomonas_E protegens
GCA_020636435.1     0.9998    S    p__Bacteroidota     f__Saprospiraceae         s__JACJXW01 sp020636435
GCA_001969245.1     0.9997    S  p__Pseudomonadota f__Burkholderiaceae_B        s__Comamonas acidovorans
GCA_903166445.1     0.9997    S  p__Pseudomonadota f__Enterobacteriaceae        s__Klebsiella pneumoniae
GCA_903166485.1     0.9997    S  p__Pseudomonadota f__Enterobacteriaceae        s__Klebsiella pneumoniae
GCA_903166415.1     0.9997    S  p__Pseudomonadota f__Enterobacteriaceae        s__Klebsiella pneumoniae
GCA_001931885.1     0.9996    S  p__Pseudomonadota f__Enterobacteriaceae         s__Citrobacter_B koseri
GCA_016745215.1     0.9991    S  p__Pseudomonadota      f__Chromatiaceae    s__Thiohalocapsa sp003525925
GCA_901456295.1     0.9990    S  p__Pseudomonadota f__Enterobacteriaceae    s__Citrobacter portucalensis
GCA_022184905.1     0.9985    S p__Cyanobacteriota     f__Microcoleaceae          s__Limnoraphis robusta
GCA_022483545.1     0.9985    S       p__Bacillota   f__Lactobacillaceae s__Lacticaseibacillus paracasei
GCA_022483205.1     0.9984    S       p__Bacillota   f__Lactobacillaceae s__Lacticaseibacillus paracasei
GCA_000738045.1     0.9958    S    p__Bacteroidota     f__Bacteroidaceae            s__Phocaeicola dorei
GCA_910577865.1     0.9957    S     p__Bacillota_A     f__Clostridiaceae          s__Sarcina perfringens
GCA_913062945.1     0.9957    S  p__Pseudomonadota       f__Vibrionaceae            s__Vibrio diabolicus
```

Pool size: **2,491 MAGs**. As expected, dominated by species (Klebsiella, Citrobacter, Phocaeicola, Sarcina, Vibrio) where assembly type is the only thing keeping the genome in the MAG bucket.

## Strict pool — top 15 by P(isolate) (uncultured genera in cultivated families)

```
      genome_id  p_isolate tier               phylum                 family           genus                     species
GCA_020636435.1     0.9998    S      p__Bacteroidota      f__Saprospiraceae     g__JACJXW01     s__JACJXW01 sp020636435
GCA_022184905.1     0.9985    S   p__Cyanobacteriota      f__Microcoleaceae  g__Limnoraphis      s__Limnoraphis robusta
GCA_020636105.1     0.9939    S      p__Bacteroidota      f__Saprospiraceae         g__BCD1         s__BCD1 sp020636105
GCA_018399035.1     0.9895    S    p__Pseudomonadota       f__Chromatiaceae    g__Thiocapsa    s__Thiocapsa sp018399035
GCA_002243285.1     0.9770    S      p__Bacteroidota   f__Crocinitomicaceae   g__Fluviicola     s__Fluviicola riflensis
GCA_020636095.1     0.9759    S      p__Bacteroidota      f__Saprospiraceae     g__JACJXW01     s__JACJXW01 sp020636095
GCA_003491285.1     0.9684    S p__Methanobacteriota f__Methanobacteriaceae       g__UBA117       s__UBA117 sp002494785
GCA_019635935.1     0.9659    S    p__Pseudomonadota     f__Parvibaculaceae g__Parvibaculum s__Parvibaculum sp019635935
GCA_023953845.1     0.9593    S    p__Pseudomonadota        f__Rhizobiaceae     g__JAFKJW01     s__JAFKJW01 sp023953845
GCA_903927965.1     0.9399    S      p__Bacteroidota  f__Prolixibacteraceae     g__JAAFHB01     s__JAAFHB01 sp903927965
GCA_012103375.1     0.9360    S    p__Pseudomonadota    f__Rhodobacteraceae       g__WLWX01       s__WLWX01 sp012103375
GCA_016713705.1     0.9288    S    p__Pseudomonadota      f__Rhodocyclaceae       g__SSSZ01       s__SSSZ01 sp016720065
GCA_016712225.1     0.9251    S      p__Bacteroidota      f__Saprospiraceae      g__UBA3362      s__UBA3362 sp016712225
GCA_024233835.1     0.9178    S     p__Spirochaetota      f__Leptospiraceae     g__JACKPC01     s__JACKPC01 sp024233835
GCA_016717265.1     0.9121    S      p__Bacteroidota      f__Saprospiraceae      g__UBA3362      s__UBA3362 sp016717265
```

**S-tier (P ≥ 0.9) family breakdown:**
```
family
f__Saprospiraceae         5
f__Crocinitomicaceae      2
f__Microcoleaceae         1
f__Chromatiaceae          1
f__Methanobacteriaceae    1
f__Parvibaculaceae        1
f__Rhizobiaceae           1
f__Prolixibacteraceae     1
f__Rhodobacteraceae       1
f__Rhodocyclaceae         1
```

Notable findings:

- **Saprospiraceae** (Bacteroidota) is the single biggest source of S-tier strict candidates — 5 of 19 — across multiple uncultured genera (`g__JACJXW01`, `g__BCD1`, `g__UBA3362`). This is a known under-cultured family with one canonical cultured member (Saprospira); the model is identifying multiple lineages with isolate-like metabolic completeness.
- **Methanobacteriaceae uncultured genus** (`g__UBA117`) — methanogen candidate.
- **`g__Fluviicola riflensis`** — Rifle aquifer Banfield-lab genome, environmentally significant subsurface lineage.
- **`g__Limnoraphis`** (Microcoleaceae, Cyanobacteria) — known but rarely cultivated freshwater genus.
- **`g__Thiocapsa`** (Chromatiaceae) — purple sulfur bacterium, anaerobic photosynthesis.

## H3 test: are S-tier candidates enriched in environmentally-interesting taxa?

Defined `env_interest` = phylum in {Acidobacteriota, Actinomycetota, Pseudomonadota, Bacillota, Bacteroidota, Chloroflexota, Verrucomicrobiota, Planctomycetota, Gemmatimonadota, Cyanobacteriota, Spirochaetota, Myxococcota} — the soil/subsurface/plant-associated phyla we care about.

**Fisher's exact: OR = 4.44, p = 3.6e-27**. S-tier strict candidates are 4.4× enriched in environmentally-interesting phyla relative to the uncultured baseline. **H3 supported.**

## Phylum-level candidate breakdown (strict pool)

![](../figures/candidate_phyla_strict.png)

Compare to the broader lenient pool:

![](../figures/candidate_phyla.png)

Pseudomonadota dominate the lenient pool but not the strict pool — most uncultured *Pseudomonadota genera* belong to families already heavily cultivated. Bacteroidota become the top phylum in the strict view, driven by the Saprospiraceae signal.

## Outputs

- `data/cultivability_candidates.tsv` — 2,491 lenient candidates
- `data/cultivability_candidates_strict.tsv` — 256 strict candidates (uncultured genera)
- `figures/candidate_phyla.png`, `figures/candidate_phyla_strict.png`

Hand-off to NB05 for anchored validation against `clay_confined_subsurface` (Mont Terri) and `oak_ridge_cultivation_gap` (Oak Ridge ORFRC) ground truth.